In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

In [ ]:
DAILY_CSV = Path("../src/Results/PF_eta1.csv")
methods = ["GMM", "EW", "MC", "DRMC", "CMC", "DRCMC", "OTCMC"]

tau = 0.1
ANN_DAYS = 252
SCALE_TO_PERCENT = True
FILTER_YEARS = []  

def parse_eta_from_path(p: Path, default_eta=None):
    stem = p.stem
    m = re.search(r"(?:^|[_\-])eta[_\-]?(\d+(?:\.\d+)?)(?=$|[_\-.])", stem, flags=re.IGNORECASE)
    if m:
        return float(m.group(1))
    if default_eta is not None:
        return float(default_eta)
    raise ValueError(f"Could not parse eta from filename: {p.name}.")

eta = parse_eta_from_path(DAILY_CSV)

if not DAILY_CSV.exists():
    raise FileNotFoundError(f"Daily CSV not found: {DAILY_CSV.resolve()}")

df = pd.read_csv(DAILY_CSV, low_memory=False)

time_col = None
for c in ["time", "date", "datetime", "timestamp"]:
    if c in df.columns:
        time_col = c
        break

if time_col is not None:
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

if FILTER_YEARS:
    if time_col is None:
        raise ValueError("FILTER_YEARS was specified, but no time/date column exists in the CSV.")
    df = df[df[time_col].dt.year.isin(FILTER_YEARS)].copy()

ret_cols = []
for m in methods:
    rc = f"ret_{m}"
    if rc in df.columns:
        df[rc] = pd.to_numeric(df[rc], errors="coerce")
        ret_cols.append(rc)

if SCALE_TO_PERCENT:
    df[ret_cols] = df[ret_cols] * 100.0

if "GMM_K" in df.columns:
    df["GMM_K"] = pd.to_numeric(df["GMM_K"], errors="coerce")
if "GMM_eps" in df.columns:
    df["GMM_eps"] = pd.to_numeric(df["GMM_eps"], errors="coerce")

def cvar_of_series(x, tau=0.1, tail="upper"):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    x_sorted = x.sort_values()
    k = int(np.ceil(len(x_sorted) * tau))
    k = max(1, k)
    if tail == "lower":
        return float(x_sorted.iloc[:k].mean())
    elif tail == "upper":
        return float(x_sorted.iloc[-k:].mean())
    else:
        raise ValueError("tail must be 'upper' or 'lower'")

def infer_ann_days(df, time_col, min_days=5):
    if time_col is None or time_col not in df.columns:
        return None
    if df[time_col].isna().all():
        return None

    t = df[time_col].dropna().dt.normalize()
    if t.nunique() < min_days:
        return None

    counts = t.groupby(t.dt.year).nunique()
    if len(counts) == 0:
        return None
    return float(counts.mean())

if ANN_DAYS is None:
    ann_factor = None
elif isinstance(ANN_DAYS, (int, float)):
    ann_factor = float(ANN_DAYS)
elif ANN_DAYS == "infer":
    ann_factor = infer_ann_days(df, time_col)
else:
    raise ValueError("ANN_DAYS must be None, a number, or 'infer'")

rows = []

for m in methods:
    ret_col = f"ret_{m}"
    if ret_col not in df.columns:
        continue

    r = df[ret_col].dropna()
    if len(r) == 0:
        continue

    mu = float(r.mean())
    sd = float(r.std(ddof=1))

    sharpe_daily = (mu / sd) if sd > 0 else np.nan
    sharpe_ann = (
        float(sharpe_daily * np.sqrt(ann_factor))
        if (ann_factor is not None and np.isfinite(sharpe_daily))
        else np.nan
    )

    L = -r
    cvar_loss = cvar_of_series(L, tau=tau, tail="upper")
    cvar_return = cvar_of_series(r, tau=tau, tail="lower")

    risk = cvar_loss - eta * mu

    row_dict = {
        "eta": eta,
        "method": m,
        "N_days": int(len(r)),
        "risk": risk,
        "mean_ret": mu,
        f"CVaR_loss_upper_tau{tau}": cvar_loss,
        "Sharpe_ann": sharpe_ann,
        "scaled_to_percent": SCALE_TO_PERCENT,
        "filter_years": str(FILTER_YEARS) if FILTER_YEARS else "ALL",
    }

    if m == "GMM":
        if "GMM_K" in df.columns and df["GMM_K"].notna().any():
            row_dict["avg_GMM_K"] = float(df["GMM_K"].dropna().mean())
        else:
            row_dict["avg_GMM_K"] = np.nan

        if "GMM_eps" in df.columns and df["GMM_eps"].notna().any():
            row_dict["avg_GMM_eps"] = float(df["GMM_eps"].dropna().mean())
        else:
            row_dict["avg_GMM_eps"] = np.nan

    rows.append(row_dict)

summary = pd.DataFrame(rows).round(3)

if len(summary) > 0:
    summary = summary.sort_values(["eta", "risk"], ascending=[True, True]).reset_index(drop=True)

display(summary)